# ⚖️ 04 - Comparaison des Modèles

Analyse comparative détaillée des différents modèles.

**M2 MoSEF - Université Paris 1 Panthéon-Sorbonne**

In [ ]:
import sys
sys.path.insert(0, '..')

import io
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm

from app.services.captcha_generator import CaptchaGenerator
from app.services.solver_service import SolverService

In [ ]:
generator = CaptchaGenerator()
solver = SolverService()
print(f"Modèles: {solver.available_models}")

## 1. Comparaison sur un échantillon

In [ ]:
# Générer des échantillons
n_samples = 20
results = []

for i in tqdm(range(n_samples)):
    image, true_text = generator.generate(length=5, noise_level=0.3)
    buffer = io.BytesIO()
    image.save(buffer, format="PNG")
    image_bytes = buffer.getvalue()
    
    compare_result = solver.compare(image_bytes, true_text=true_text)
    
    for model, res in compare_result.results.items():
        results.append({
            'sample': i,
            'true_text': true_text,
            'model': model,
            'prediction': res.text,
            'correct': res.text.lower() == true_text.lower(),
            'confidence': res.confidence,
            'time_ms': res.processing_time_ms,
        })

df = pd.DataFrame(results)
df.head(10)

## 2. Accuracy par modèle

In [ ]:
accuracy_df = df.groupby('model')['correct'].mean() * 100

plt.figure(figsize=(10, 5))
accuracy_df.sort_values(ascending=False).plot(kind='bar', color='steelblue')
plt.title('Accuracy par modèle')
plt.ylabel('Accuracy (%)')
plt.xlabel('Modèle')
plt.xticks(rotation=45)
plt.ylim(0, 100)
for i, v in enumerate(accuracy_df.sort_values(ascending=False)):
    plt.text(i, v + 2, f'{v:.1f}%', ha='center')
plt.tight_layout()
plt.show()

## 3. Temps d'exécution

In [ ]:
plt.figure(figsize=(10, 5))
sns.boxplot(data=df, x='model', y='time_ms')
plt.title('Distribution du temps d\'exécution par modèle')
plt.ylabel('Temps (ms)')
plt.xlabel('Modèle')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 4. Résumé

In [ ]:
summary = df.groupby('model').agg({
    'correct': ['sum', 'count', 'mean'],
    'time_ms': ['mean', 'std'],
    'confidence': 'mean'
}).round(2)

summary.columns = ['Corrects', 'Total', 'Accuracy', 'Temps_moy', 'Temps_std', 'Confiance_moy']
summary['Accuracy'] = (summary['Accuracy'] * 100).round(1).astype(str) + '%'
summary

---
**Fin du notebook 04**